# ExhibitFlow — chạy pipeline trên Google Colab

Notebook chỉ **clone repo (nhánh `Hung`) → cài thư viện → chạy `main.py`**.

**Chuẩn bị:**
1. *Runtime → Change runtime type* → GPU **G4** (mạnh nhất) hoặc **A100** (dự phòng). TPU không dùng được.
2. Đưa **trọng số** (`best_ckpt.pth`, `osnet_x0_25.pth`, `ltc_motion_mot17.pth`, `dma_gbm_mot17.gbm`) và **video** lên Google Drive
   (hoặc upload thẳng vào `/content`). Đặt đường dẫn ở ô cấu hình bên dưới.
3. Tuỳ chọn: secret `HF_TOKEN` (Colab → biểu tượng chìa khoá) để tải model nhanh hơn.

## 1. GPU + Google Drive

In [ ]:
import subprocess
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
if smi.returncode != 0 or not smi.stdout.strip():
    raise RuntimeError('Không thấy GPU NVIDIA. Runtime -> Change runtime type -> chọn GPU (G4 / A100 / L4 / T4). TPU không chạy được llama.cpp.')
print('GPU:', smi.stdout.strip())

MOUNT_DRIVE = True          # False nếu bạn upload thẳng file vào /content
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
try:                                                    # HF_TOKEN từ Colab secrets (nếu có)
    from google.colab import userdata
    import os
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

## 2. Cấu hình

In [ ]:
REPO_URL = 'https://github.com/khangkaka066/ExhibitFlow.git'
BRANCH = 'Hung'
CLONE_DIR = '/content/ExhibitFlow'

VIDEO_DIRS = ['/content/drive/MyDrive/exhibitflow/videos']       # thư mục chứa video
VIDEO_PATHS = []                                                 # hoặc liệt kê từng file
WEIGHTS_DIRS = ['/content/drive/MyDrive/exhibitflow/weights']    # thư mục chứa 4 file trọng số
CONFIG_FILES = []            # vd ['configs/default.yaml']
EXTRA_ARGS = []              # vd ['--set', 'models.vlm.max_side=768', '--set', 'models.vlm.n_images=12']
MAX_FRAMES = None            # vd 300 để chạy thử nhanh
SAVE_VIDEO = False           # True: ghi live.mp4

WORK_DIR = '/content/exhibitflow_work'
OUT_DIR = '/content/exhibitflow_results'
ZIP_BASE = '/content/exhibitflow_results'

## 3. Lấy mã nguồn

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

ROOT = Path(CLONE_DIR)
PROJ = ROOT / 'exhibitflow_pipeline'


def sh(cmd, cwd=None):
    print('$', ' '.join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)


# Lấy mã nguồn: chỉ tải thư mục exhibitflow_pipeline của nhánh BRANCH (repo có nhiều file lớn khác).
if (ROOT / '.git').exists():
    sh(['git', 'fetch', '--depth', '1', 'origin', BRANCH], cwd=ROOT)
    sh(['git', 'reset', '--hard', 'FETCH_HEAD'], cwd=ROOT)
else:
    try:
        sh(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--filter=blob:none', '--sparse', REPO_URL, ROOT])
        sh(['git', 'sparse-checkout', 'set', 'exhibitflow_pipeline'], cwd=ROOT)
    except subprocess.CalledProcessError:          # git cũ không có --sparse -> clone thường
        subprocess.run(['rm', '-rf', str(ROOT)])
        sh(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, ROOT])
print('Commit:', subprocess.run(['git', 'log', '-1', '--format=%h %s (%cd)'], cwd=ROOT, capture_output=True, text=True).stdout.strip())
assert (PROJ / 'main.py').exists(), f'Không thấy {PROJ}/main.py: nhánh {BRANCH} đã có thư mục exhibitflow_pipeline chưa?'


def run_main(*args):
    """Chạy main.py, in log ngay khi có. Trả mã thoát."""
    cmd = [sys.executable, '-u', 'main.py', *map(str, args)]
    print('$', ' '.join(cmd))
    p = subprocess.Popen(cmd, cwd=PROJ, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                         errors='replace', bufsize=1, env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
    for line in p.stdout:
        print(line, end='')
    return p.wait()


def common_args():
    a = []
    for w in WEIGHTS_DIRS:
        a += ['--weights-dir', w]
    a += ['--work-dir', WORK_DIR]
    for c in CONFIG_FILES:
        a += ['--config', c]
    return a + list(EXTRA_ARGS)

## 4. Cài thư viện (~2-4 phút)

In [ ]:
sh([sys.executable, PROJ / 'scripts' / 'install_deps.py'])

## 5. Kiểm tra môi trường + trọng số

In [ ]:
rc = run_main('--check', *common_args())
print('\nmã thoát:', rc, '(0 = sẵn sàng, 2 = thiếu thành phần: xem dòng !!! ở trên)')

## 6. Chạy pipeline

In [ ]:
args = ['--out', OUT_DIR, *common_args()]
if VIDEO_PATHS:
    args += ['--videos', *VIDEO_PATHS]
if VIDEO_DIRS:
    for d in VIDEO_DIRS:
        args += ['--video-dir', d]
if MAX_FRAMES:
    args += ['--max-frames', MAX_FRAMES]
if SAVE_VIDEO:
    args += ['--save-video']
t0 = time.time()
rc = run_main(*args)
print(f'\nmã thoát {rc} sau {(time.time() - t0) / 60:.1f} phút  (0 = ổn, 1 = có video lỗi, 2 = tham số / môi trường sai)')

## 7. Xem kết quả

In [ ]:
import pandas as pd
from IPython.display import display

out = Path(OUT_DIR)
summary = json.loads((out / 'summary.json').read_text(encoding='utf-8'))
print(f"{summary['n_videos']} video | {summary['n_alerts']} cảnh báo | {summary['total_s']} s | lỗi: {len(summary['errors'])}")
display(pd.DataFrame(summary['videos']))
rows = []
for v in summary['videos']:
    res = json.loads((out / v['result']).read_text(encoding='utf-8'))
    for a in res['alerts']:
        rows.append(dict(video=v['key'], **{k: a[k] for k in ('track_id', 'artifact_id', 'behavior', 'confidence', 'alarm_start_time_s',
                                                              'alarm_end_time_s', 'evidence')}))
    p = res['performance']
    print(f"{v['key']}: {p['pipeline_fps']} FPS | VLM {p['vlm']['infer_mean_s']} s / đoạn | e2e {p['e2e_ms']['mean']} ms")
print('\nCảnh báo:')
display(pd.DataFrame(rows) if rows else 'không có cảnh báo')
for e in summary['errors']:
    print('LỖI', e['video'], e['error'][-800:])

## 8. Nén + tải kết quả về máy

In [ ]:
import shutil
zip_path = shutil.make_archive(ZIP_BASE, 'zip', OUT_DIR)
print('Đã nén:', zip_path, f'({os.path.getsize(zip_path) / 1e6:.1f} MB)')

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass